# Chrome / Vivaldi Extension (`chrome-ext/`)

> Manifest V3 extension that redirects Solveit's `esm.sh/modern-monaco` and `esm.sh/tm-grammars` requests to lightweight local no-op stubs before Monaco downloads or boots.

Solveit injects `CRAFT.js` after its own Monaco boot script, so `CRAFT.js` alone can only dispose Monaco post-load. This optional browser extension intercepts the two `esm.sh` requests at the network layer (scoped strictly to `*.solve.it.com`), saving ~61 MB of tab RAM per dialog with zero background process. Run `uv run craft-cdp build-ext` (or `uv run craft-cdp build --sync`) to regenerate `chrome-ext/` from this notebook.

In [ ]:
#| default_exp chrome_ext

In [ ]:
#| export
import json
from fastcore.utils import *

from craft_dev.auth import REPO_ROOT
from craft_dev.build import _check_js_syntax

DEFAULT_CHROME_EXT_DIR = REPO_ROOT/'chrome-ext'

## Manifest, Redirect Rules & Stubs

The extension uses Chrome's built-in `declarativeNetRequest` static ruleset (`rules.json`) with no background service worker. Both rules restrict `initiatorDomains` to `["solve.it.com"]` so `esm.sh` requests on any other site are untouched.

In [ ]:
#| export
EXT_MANIFEST = {
    'manifest_version': 3,
    'name': 'Solveit CRAFT — Block Monaco',
    'version': '0.1.0',
    'description': 'Redirects Solveit modern-monaco and tm-grammars requests on *.solve.it.com to local no-op stubs so CRAFT uses CodeMirror 6 without loading Monaco.',
    'permissions': ['declarativeNetRequest'],
    'host_permissions': ['https://*.solve.it.com/*', 'https://esm.sh/*'],
    'declarative_net_request': {
        'rule_resources': [{
            'id': 'block_monaco',
            'enabled': True,
            'path': 'rules.json',
        }],
    },
    'web_accessible_resources': [{
        'resources': ['monaco_stub.mjs', 'empty_grammar.json'],
        'matches': ['https://*.solve.it.com/*'],
    }],
}

EXT_RULES = [
    {
        'id': 1,
        'priority': 1,
        'action': {'type': 'redirect', 'redirect': {'extensionPath': '/monaco_stub.mjs'}},
        'condition': {
            'urlFilter': '||esm.sh/modern-monaco',
            'initiatorDomains': ['solve.it.com'],
            'resourceTypes': ['script', 'xmlhttprequest', 'other'],
        },
    },
    {
        'id': 2,
        'priority': 1,
        'action': {'type': 'redirect', 'redirect': {'extensionPath': '/empty_grammar.json'}},
        'condition': {
            'urlFilter': '||esm.sh/tm-grammars',
            'initiatorDomains': ['solve.it.com'],
            'resourceTypes': ['xmlhttprequest', 'script', 'other'],
        },
    },
]

EMPTY_GRAMMAR_JSON = {'name': 'python', 'scopeName': 'source.python', 'patterns': []}

`MONACO_STUB_JS` implements the minimal ES module surface that Solveit's `monedit.js` and `core-*.js` touch during boot and cell swaps, backed by `Proxy` no-op fallbacks so Monaco's web workers and grammars are never fetched.

In [ ]:
#| export
MONACO_STUB_JS = r"""// Lightweight no-op modern-monaco stub generated from nbs/05_chrome_ext.ipynb
const noop = () => {};
const disp = () => ({ dispose: noop });
const makeProxy = base => new Proxy(base, {
  get(t, k) {
    if (k in t) return t[k];
    if (typeof k === 'symbol') return undefined;
    return k.startsWith('onDid') || k.startsWith('onWill') || k.startsWith('register') ? disp : noop;
  }
});

class Uri {
  constructor(s = '') { this._s = String(s); }
  toString() { return this._s; }
  static parse(s) { return new Uri(s); }
  static file(s) { return new Uri('file://' + s); }
}

class Position {
  constructor(lineNumber = 1, column = 1) { this.lineNumber = lineNumber; this.column = column; }
}

class Range {
  constructor(sl = 1, sc = 1, el = 1, ec = 1) {
    this.startLineNumber = sl; this.startColumn = sc;
    this.endLineNumber = el; this.endColumn = ec;
  }
}

const makeModel = (val = '', lang = 'plaintext', uri = null) => {
  let _val = String(val ?? ''), _lang = lang;
  const _uri = uri || Uri.parse('inmemory://craft-stub');
  return makeProxy({
    __craftStub: true,
    uri: _uri,
    getValue: () => _val,
    setValue: v => { _val = String(v ?? ''); },
    getLanguageId: () => _lang,
    setLanguage: l => { _lang = l; },
    getLineCount: () => _val ? _val.split('\n').length : 1,
    getFullModelRange: () => new Range(1, 1, 1, 1),
    pushEditOperations: () => null,
    isDisposed: () => false,
    dispose: noop,
  });
};

const makeEditor = () => {
  let _model = null;
  const _node = typeof document !== 'undefined' ? document.createElement('div') : null;
  return makeProxy({
    __craftStub: true,
    getContainerDomNode: () => _node,
    getDomNode: () => _node,
    getModel: () => _model,
    setModel: m => { _model = m; },
    getValue: () => _model?.getValue?.() || '',
    setValue: v => { _model?.setValue?.(v); },
    getPosition: () => new Position(1, 1),
    getSelection: () => new Range(1, 1, 1, 1),
    getRawOptions: () => ({}),
    getOption: () => undefined,
    hasTextFocus: () => false,
    hasWidgetFocus: () => false,
    layout: noop,
    focus: noop,
    updateOptions: noop,
    trigger: noop,
    dispose: noop,
  });
};

export const editor = makeProxy({
  __craftStub: true,
  create: () => makeEditor(),
  createDiffEditor: () => makeEditor(),
  createModel: (val, lang, uri) => makeModel(val, lang, uri),
  getModel: () => null,
  getEditors: () => [],
  getDiffEditors: () => [],
  getModels: () => [],
  setModelLanguage: (m, l) => m?.setLanguage?.(l),
  setTheme: noop,
  defineTheme: noop,
  remeasureFonts: noop,
  addKeybindingRule: disp,
  registerLinkOpener: disp,
  registerEditorOpener: disp,
});

export const languages = makeProxy({
  __craftStub: true,
  getLanguages: () => [],
  onLanguage: disp,
});

export const KeyMod = { CtrlCmd: 2048, Shift: 1024, Alt: 512, WinCtrl: 256 };
export const KeyCode = new Proxy({}, { get: (_, k) => (typeof k === 'symbol' ? undefined : 0) });
export { Uri, Range, Position };

export const monaco = { __craftStub: true, editor, languages, Uri, Range, Position, KeyMod, KeyCode };
globalThis.monaco = monaco;

export async function init() { return monaco; }
export default monaco;
"""

## Building `chrome-ext/`

`build_chrome_ext` writes all four static extension files into `out_dir` (default: `chrome-ext/` at the repository root) and validates `monaco_stub.mjs` with `node --check`.

In [ ]:
#| export
def build_chrome_ext(
    out_dir=DEFAULT_CHROME_EXT_DIR, # Output directory for the unpacked Chrome/Vivaldi extension
    check_js:bool=True, # Validate monaco_stub.mjs with `node --check`
):
    "Generate the unpacked Manifest V3 Chrome/Vivaldi extension that stubs Solveit's Monaco requests."
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    files = {
        'manifest.json': json.dumps(EXT_MANIFEST, indent=2) + '\n',
        'rules.json': json.dumps(EXT_RULES, indent=2) + '\n',
        'monaco_stub.mjs': MONACO_STUB_JS,
        'empty_grammar.json': json.dumps(EMPTY_GRAMMAR_JSON, indent=2) + '\n',
    }
    written = {}
    for name, content in files.items():
        p = out_dir/name
        p.mk_write(content)
        written[name] = p.stat().st_size
    if check_js: _check_js_syntax(out_dir/'monaco_stub.mjs')
    return written

Verify `build_chrome_ext` writes valid MV3 JSON, keeps every redirect rule scoped to `solve.it.com`, exposes each redirect target in `web_accessible_resources`, and passes `node --check`.

In [ ]:
import tempfile
from fastcore.test import test_eq

with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    rep = build_chrome_ext(out_dir=td, check_js=True)
    test_eq(sorted(rep), ['empty_grammar.json', 'manifest.json', 'monaco_stub.mjs', 'rules.json'])
    man = (td/'manifest.json').read_json()
    rules = (td/'rules.json').read_json()
    test_eq(man['manifest_version'], 3)
    war = set(man['web_accessible_resources'][0]['resources'])
    for r in rules:
        test_eq(r['condition']['initiatorDomains'], ['solve.it.com'])
        target = r['action']['redirect']['extensionPath'].lstrip('/')
        assert target in war and (td/target).is_file(), f"Missing redirect target {target}"